In [4]:
import os
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_customers_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_orders_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_products_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/product_category_name_translation.csv


In [5]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv")
df = df[["review_score", "review_comment_message"]].dropna()

print(df.shape)
print(df["review_score"].value_counts().sort_index())
df.sample(5, random_state=1)

(40977, 2)
review_score
1     8745
2     2145
3     3557
4     5976
5    20554
Name: count, dtype: int64


,review_score,review_comment_message
47666,5,Produto de ótima qualidade.. fiquei muito sati...
51647,5,"Muito bom,super indico,entrega muito rápida pa..."
3044,5,entregou muito antes do prazo estimado.
6247,5,"Produto entregue, testado e aprovado!"
76468,5,Ótimo produto excelente qualidade entrega rápida.


## Problema
Classificar automaticamente avaliações de clientes como negativas (nota 1 ou 2)
para ajudar a priorizar o atendimento.

## Métrica principal
Recall da classe negativa: deixar passar uma reclamação custa mais do que revisar
uma avaliação a mais. Também acompanho precisão e F1.

## Limitação
A nota é um rótulo aproximado de reclamação. Nem toda nota baixa é um problema
do serviço, e nem toda reclamação vem com nota baixa.

In [6]:
df["negativa"] = (df["review_score"] <= 2).astype(int)

print(df["negativa"].value_counts())
print("Proporção de negativas:", round(df["negativa"].mean(), 3))

negativa
0    30087
1    10890
Name: count, dtype: int64
Proporção de negativas: 0.266


In [7]:
pd.set_option("display.max_colwidth", 200)
df[df["negativa"] == 1].sample(8, random_state=3)[["review_score", "review_comment_message"]]

,review_score,review_comment_message
21026,1,"Meu produto não foi entregue ainda, espero que seja!"
78669,1,Ainda não recebi.
55067,1,"Nao recebi os produtos e nem o ressarcimento do valor pago. Estou reclamando e não estão me dando o devido retorno, uma completa falta de consideração pelo cliente. Estarei procurando os meus dire..."
46349,1,A mercadoria ainda não chegou.
81309,1,"Produto não é original, e não era isso que tinha no anúncio. Quero devolver o produto e o estorno do pagamento"
98560,2,"Comprei três luminárias pendentes, recebi apenas duas."
81210,1,Não recebi meu produto ainda
41569,1,não entregaram a mercadoria completa


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["review_comment_message"], df["negativa"],
    test_size=0.2, random_state=42, stratify=df["negativa"]
)

print("Treino:", len(X_train), "| Teste:", len(X_test))
print("Negativas no treino:", round(y_train.mean(), 3))
print("Negativas no teste:", round(y_test.mean(), 3))

Treino: 32781 | Teste: 8196
Negativas no treino: 0.266
Negativas no teste: 0.266


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

baseline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)

print(classification_report(y_test, pred, target_names=["não negativa", "negativa"]))

              precision    recall  f1-score   support

não negativa       0.97      0.88      0.92      6018
    negativa       0.73      0.93      0.82      2178

    accuracy                           0.89      8196
   macro avg       0.85      0.90      0.87      8196
weighted avg       0.91      0.89      0.89      8196



In [10]:
from sklearn.svm import LinearSVC

svm = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000),
    LinearSVC(class_weight="balanced")
)

svm.fit(X_train, y_train)
pred_svm = svm.predict(X_test)

print(classification_report(y_test, pred_svm, target_names=["não negativa", "negativa"]))

              precision    recall  f1-score   support

não negativa       0.95      0.90      0.92      6018
    negativa       0.76      0.88      0.81      2178

    accuracy                           0.89      8196
   macro avg       0.86      0.89      0.87      8196
weighted avg       0.90      0.89      0.90      8196



In [11]:
erros = pd.DataFrame({
    "texto": X_test,
    "real": y_test,
    "previsto": pred
})

pd.set_option("display.max_colwidth", 200)

# Reclamações que o modelo deixou passar (o erro mais caro)
print("RECLAMAÇÕES NÃO DETECTADAS")
display(erros[(erros["real"] == 1) & (erros["previsto"] == 0)].sample(12, random_state=1))

RECLAMAÇÕES NÃO DETECTADAS


,texto,real,previsto
471,No dia 19 de maio usei e a à alça quebrou sem muito peso,1,0
2072,recomendo a loa pois cumpriu o prazo certinho mas o produto é muito ruim...pessima qualidade.,1,0
37051,"utilizo o site das amaericanas a muitos anos, sempre tive um bom retorno, as vezes demora,mas os produtos chegam sempre embalados. mas dessa vez não recebi, por isso a avaliação negativa.",1,0
18062,Recebi o pedido em 8 dias.,1,0
45509,Sai mto rapido,1,0
68162,Nós compramos 5 pufs mas só foram entregues 4,1,0
15115,"material com uma qualidade muito boa,letra para leitura muito boa,e foi entregue tudo certo.",1,0
45728,Gostei muito,1,0
85278,O relógio não veio com manual de instruções.,1,0
15369,mas poderia ter sido mais rapido,1,0


In [12]:
# Falsos alarmes: o modelo disse "reclamação", mas a nota era 3 a 5
print("FALSOS ALARMES")
display(erros[(erros["real"] == 0) & (erros["previsto"] == 1)].sample(12, random_state=1))

FALSOS ALARMES


,texto,real,previsto
95975,A capa veio com as tiras abaixo do local indicado na foto.,0,1
82657,O produto nunca apareceu em casa!,0,1
68771,P Correio ainda não me entregou o produto,0,1
33843,Produto com baixa qualidade.,0,1
90028,Não chegou o meu produto. Quero saber o que aconteceu?,0,1
66645,"Comprei pra uma criança de um ano, é uma gracinha e do tamanho dele.",0,1
55171,"mas não serviu na minha impressora - HP OFFICEJET J4660 - PRECISO TROCAR ,, AGUARDO",0,1
50139,"Comprei 2 rolos de cabo 4mm e até agora so recebi 1, achei estranho porque na nota fiscal veio constando 2 e so recebi 1. Espero não ter dor de cabeça. Nao vou recmendar a loja ate resolver a situ...",0,1
6807,Preciso que vcs resolvam o meu problema de trocar o produto,0,1
85481,Se o produto tá dentro do prazo de entrega não precisava a empresa informar que o teve problema com o correios. É resolver o problema dentro do prazo sem precisar preocupar o cliente.,0,1
